## Applies all corrections identified in 01_profiling_and_dq.ipynb and engineers key analytical columns needed for metric construction and hypothesis testing. Cleaned datasets are saved to disk and used as the single source of truth for all downstream notebooks.

#### *01. Correcting casing inconsistency in dim_products.category. Source data contains beverages in lowercase while all other categories use Title Case. Checking affected row count before correction, then verifying after.*



In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
#Loading CSV files

DATA_DIR = r"E:\Portfolio_Projects\Supply_Chain_FMCG\data"

dim_customers      = pd.read_csv(os.path.join(DATA_DIR, 'dim_customers.csv'))
dim_products       = pd.read_csv(os.path.join(DATA_DIR, 'dim_products.csv'))
dim_date           = pd.read_csv(os.path.join(DATA_DIR, 'dim_date.csv'))
dim_targets_orders = pd.read_csv(os.path.join(DATA_DIR, 'dim_targets_orders.csv'))
fact_order_lines   = pd.read_csv(os.path.join(DATA_DIR, 'fact_order_lines.csv'))
fact_orders_agg    = pd.read_csv(os.path.join(DATA_DIR, 'fact_orders_aggregate.csv'))

In [3]:
#Re-applying the standardizations identified in session 01_profiling_and_dq

fact_order_lines.rename(columns={
    'delivery_qty':    'delivered_qty',
    'In Full':         'in_full_line',
    'On Time':         'on_time_line',
    'On Time In Full': 'otif_line'
}, inplace=True)

dim_date['date']   = pd.to_datetime(dim_date['date'],   format='%d-%b-%y')
dim_date['mmm_yy'] = pd.to_datetime(dim_date['mmm_yy'], format='%d-%b-%y')

fact_order_lines['order_placement_date'] = pd.to_datetime(
    fact_order_lines['order_placement_date'], format='%A, %B %d, %Y')
fact_order_lines['agreed_delivery_date'] = pd.to_datetime(
    fact_order_lines['agreed_delivery_date'], format='%A, %B %d, %Y')
fact_order_lines['actual_delivery_date'] = pd.to_datetime(
    fact_order_lines['actual_delivery_date'], format='%A, %B %d, %Y')

fact_orders_agg['order_placement_date'] = pd.to_datetime(
    fact_orders_agg['order_placement_date'], format='%d-%b-%y')

In [4]:
#Checking before fix

print("BEFORE FIX:")
print(dim_products['category'].value_counts())

BEFORE FIX:
category
Dairy        12
Food          3
beverages     3
Name: count, dtype: int64


In [5]:
#Applying changes

dim_products['category'] = dim_products['category'].str.strip().str.title()

In [6]:
#Verifying after fix

print("\nAFTER FIX:")
print(dim_products['category'].value_counts())


AFTER FIX:
category
Dairy        12
Food          3
Beverages     3
Name: count, dtype: int64


#### *02. Engineering three analytical columns that power every hypothesis test and diagnostic analysis downstream. These columns do not exist in the raw data but are derivable from it -> promise window captures lead time commitment, days late captures delivery failure magnitude, and shortfall captures supply failure magnitude.*

__Feature engineering__

- Promise window

In [7]:
# Number of days AtliQ promised between order placement and delivery

fact_order_lines['promise_window'] = (
    fact_order_lines['agreed_delivery_date'] -
    fact_order_lines['order_placement_date']
).dt.days

- Days late

In [8]:
# Positive = late, Zero = on time, Negative = early

fact_order_lines['days_late'] = (
    fact_order_lines['actual_delivery_date'] -
    fact_order_lines['agreed_delivery_date']
).dt.days

- Shortfall quantity and percentage

In [9]:
fact_order_lines['shortfall_qty'] = (
    fact_order_lines['order_qty'] -
    fact_order_lines['delivered_qty']
)

fact_order_lines['shortfall_pct'] = (
    fact_order_lines['shortfall_qty'] /
    fact_order_lines['order_qty'] * 100
).round(2)

In [10]:
#Verifying the measures

print("ENGINEERED COLUMNS — SUMMARY STATS\n")
print(fact_order_lines[
    ['promise_window', 'days_late', 'shortfall_qty', 'shortfall_pct']
].describe().round(2))

ENGINEERED COLUMNS — SUMMARY STATS

       promise_window  days_late  shortfall_qty  shortfall_pct
count        57096.00   57096.00       57096.00       57096.00
mean             2.00       0.42           8.02           3.41
std              0.82       0.94          16.48           5.75
min              1.00      -1.00           0.00           0.00
25%              1.00       0.00           0.00           0.00
50%              2.00       0.00           0.00           0.00
75%              3.00       1.00           9.00           5.06
max              3.00       3.00         100.00          21.74


### AtliQ operates on a 1–3 day promise window with zero tolerance built in. Any disruption; supplier, warehouse, or transport -> has no buffer to absorb it before it becomes a metric failure. The system is structurally fragile by design.

#### *03. Joining all dimension tables to fact_order_lines to create a single analysis-ready master dataframe. All subsequent EDA, hypothesis testing, and root cause analysis will operate on this joined table rather than running repeated joins across notebooks.*

__Building master dataframe__

In [11]:
#Joining dimensions to fact_order_lines

master = fact_order_lines.merge(
    dim_customers, on='customer_id', how='left'
).merge(
    dim_products, on='product_id', how='left'
).merge(
    dim_targets_orders, on='customer_id', how='left'
)

In [13]:
#Adding month and week columns for time-based analysis

master['order_month'] = master['order_placement_date'].dt.to_period('M')
master['order_week']  = master['order_placement_date'].dt.isocalendar().week
master['day_of_month']= master['order_placement_date'].dt.day

In [14]:
#Verifying shape and columns

print(f"Master dataframe shape: {master.shape}")
print(f"\nColumns:\n{master.columns.tolist()}")
print(f"\nSample row:")
print(master.head(1).T)

Master dataframe shape: (57096, 25)

Columns:
['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty', 'agreed_delivery_date', 'actual_delivery_date', 'delivered_qty', 'in_full_line', 'on_time_line', 'otif_line', 'promise_window', 'days_late', 'shortfall_qty', 'shortfall_pct', 'customer_name', 'city', 'product_name', 'category', 'ontime_target%', 'infull_target%', 'otif_target%', 'order_month', 'order_week', 'day_of_month']

Sample row:
                                        0
order_id                      FMR34203601
order_placement_date  2022-03-01 00:00:00
customer_id                        789203
product_id                       25891601
order_qty                             110
agreed_delivery_date  2022-03-04 00:00:00
actual_delivery_date  2022-03-04 00:00:00
delivered_qty                         110
in_full_line                            1
on_time_line                            1
otif_line                               1
promise_window                   

#### *04. Saving all cleaned and engineered dataframes to the /cleaned directory. All downstream notebooks load exclusively from this folder. Raw files in /data remain untouched and unmodified throughout the project.*

__Save cleaned dataframes to disk__

In [15]:
CLEAN_DIR = r"E:\Portfolio_Projects\Supply_Chain_FMCG\cleaned"

- Dimension tables

In [16]:
dim_customers.to_csv(
    os.path.join(CLEAN_DIR, 'dim_customers.csv'), index=False)
dim_products.to_csv(
    os.path.join(CLEAN_DIR, 'dim_products.csv'), index=False)
dim_date.to_csv(
    os.path.join(CLEAN_DIR, 'dim_date.csv'), index=False)
dim_targets_orders.to_csv(
    os.path.join(CLEAN_DIR, 'dim_targets_orders.csv'), index=False)

- Fact tables

In [17]:
fact_order_lines.to_csv(
    os.path.join(CLEAN_DIR, 'fact_order_lines.csv'), index=False)
fact_orders_agg.to_csv(
    os.path.join(CLEAN_DIR, 'fact_orders_agg.csv'), index=False)

- Master dataframe

In [18]:
master.to_csv(
    os.path.join(CLEAN_DIR, 'master.csv'), index=False)

In [19]:
#Verifying the saved files 

saved_files = os.listdir(CLEAN_DIR)
print("FILES SAVED TO /cleaned:\n")
for f in sorted(saved_files):
    filepath = os.path.join(CLEAN_DIR, f)
    size_kb = os.path.getsize(filepath) / 1024
    print(f"  {f:40s}  {size_kb:>8.1f} KB")

FILES SAVED TO /cleaned:

  dim_customers.csv                              1.0 KB
  dim_date.csv                                   5.0 KB
  dim_products.csv                               0.5 KB
  dim_targets_orders.csv                         0.6 KB
  fact_order_lines.csv                        4879.0 KB
  fact_orders_agg.csv                         1184.4 KB
  master.csv                                  8401.5 KB
